# The Memory Wall y dictamen hacia Spark · Hito 2
Este cuaderno corre aparte del EDA para empezar con la memoria vacía: cada prueba se ejecuta en un
proceso hijo con un límite de memoria de 6 GB (simula una laptop de 8 GB con el sistema operativo abierto).

In [1]:

import os, json, subprocess, sys, time, warnings
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
warnings.filterwarnings("ignore")
RAIZ = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
RAW, SIN = f"{RAIZ}/data/raw", f"{RAIZ}/data/sintetico"
TAB, FIG = f"{RAIZ}/resultados/tablas", f"{RAIZ}/resultados/figuras"
os.makedirs(TAB, exist_ok=True); os.makedirs(FIG, exist_ok=True)
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": "black",
                     "axes.prop_cycle": plt.cycler(color=["black", "0.45", "0.7"])})
HOY = pd.Timestamp("2026-11-22")
RES = {}                                   # resumen de cifras que usa el informe
def guardar(df, nombre): df.to_csv(f"{TAB}/{nombre}.csv", index=False); return df
print(pd.__version__, np.__version__)


3.0.5 2.5.3


## 5. The Memory Wall: RAM en reposo vs. disco y colapso de pandas

In [2]:

import gc
for _n in ["I", "S", "S2", "P", "M", "R", "PG", "MU", "A", "A2", "B", "V", "r1", "r1all", "nat", "own", "w", "chk", "p48", "mm", "impp", "mul"]:
    globals().pop(_n, None)
gc.collect(); print("memoria liberada para las pruebas")


memoria liberada para las pruebas


In [3]:

def correr(args):
    r = subprocess.run([sys.executable, f"{RAIZ}/src/memory_wall.py"] + args, capture_output=True, text=True, cwd=RAIZ)
    return json.loads(r.stdout.strip().splitlines()[-1])
med = []
for t in ["vehiculo", "propietario", "matricula", "revision_vehicular", "pagos", "multas"]:
    for obj in (True, False):
        med.append(correr(["medir", "--csv", f"{SIN}/{t}.csv", "--limite-gb", "6"] + (["--objeto"] if obj else [])))
med.append(correr(["medir", "--csv", f"{RAW}/inec_vehiculos_matriculados_2019.csv", "--sep", ";", "--enc", "utf-8-sig", "--objeto"]))
mw = guardar(pd.DataFrame(med), "05_memory_wall_tablas")
mw[["archivo", "motor", "filas", "disco_mb", "ram_deep_mb", "ratio_deep", "rss_pico_mb", "ratio_pico", "ram_opt_mb"]]


,archivo,motor,filas,disco_mb,ram_deep_mb,ratio_deep,rss_pico_mb,ratio_pico,ram_opt_mb
0,vehiculo.csv,object,2821910,542.5,2342.0,4.32,2438.3,4.49,637.2
1,vehiculo.csv,pyarrow-str,2821910,542.5,801.9,1.48,2436.3,4.49,289.9
2,propietario.csv,object,2211545,344.3,930.9,2.70,1074.2,3.12,242.0
3,propietario.csv,pyarrow-str,2211545,344.3,396.8,1.15,1074.1,3.12,153.5
4,matricula.csv,object,2821910,292.8,1075.4,3.67,992.7,3.39,56.5
5,matricula.csv,pyarrow-str,2821910,292.8,393.0,1.34,992.4,3.39,56.5
6,revision_vehicular.csv,object,2565592,154.0,502.2,3.26,709.6,4.61,43.6
7,revision_vehicular.csv,pyarrow-str,2565592,154.0,226.0,1.47,709.8,4.61,43.6
8,pagos.csv,object,10319849,704.6,2199.4,3.12,2341.2,3.32,206.4
9,pagos.csv,pyarrow-str,10319849,704.6,930.1,1.32,2341.5,3.32,206.4


In [4]:

ob = mw[(mw.motor == "object") & (mw.archivo != "inec_vehiculos_matriculados_2019.csv")]
ar = mw[(mw.motor == "pyarrow-str")]
RES["mw"] = {"disco_total_mb": round(ob.disco_mb.sum(), 1),
             "ram_obj_total_mb": round(ob.ram_deep_mb.sum(), 1), "ratio_obj": round(ob.ram_deep_mb.sum() / ob.disco_mb.sum(), 2),
             "ram_arrow_total_mb": round(ar.ram_deep_mb.sum(), 1), "ratio_arrow": round(ar.ram_deep_mb.sum() / ar.disco_mb.sum(), 2),
             "pico_max_ratio": round(ob.ratio_pico.max(), 2), "ram_opt_total_mb": round(ob.ram_opt_mb.sum(), 1)}
col = correr(["colapso", "--dir", SIN, "--limite-gb", "6"])
RES["colapso"] = col
pd.DataFrame(col["pasos"])


,paso,rss_pico_mb,seg
0,leer vehiculo,2437.2,13.1
1,leer matricula,2437.2,16.5
2,vehiculo + matricula,2437.2,16.5
3,leer pagos,3762.4,23.2
4,MemoryError,4106.8,25.2


In [5]:

# Spark local con SOLO 2 GB para el driver: el mismo join + groupby
import psutil, threading
pico = [0]
def vigilar(p):
    while p.poll() is None:
        try:
            pr = psutil.Process(p.pid); rss = pr.memory_info().rss + sum(c.memory_info().rss for c in pr.children(recursive=True))
            pico[0] = max(pico[0], rss)
        except Exception: pass
        time.sleep(.5)
t = time.time()
proc = subprocess.Popen([sys.executable, f"{RAIZ}/spark/fase1_spark_proceso_ant.py", "--base", SIN, "--memoria", "2g"],
                        stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True, cwd=RAIZ)
th = threading.Thread(target=vigilar, args=(proc,)); th.start(); salida = proc.communicate()[0]; th.join()
sp = json.loads([l for l in salida.splitlines() if l.startswith("{")][-1]); sp["rss_pico_mb"] = round(pico[0] / 1e6)
RES["spark"] = sp; sp


{'resultado': 'COMPLETO',
 'grupos': 53852,
 'filas_union': 10886481,
 'particiones_pagos': 6,
 'seg': 67.6,
 'top_loja': [['LOJA', 4467691.56],
  ['PUYANGO', 193155.93],
  ['MACARA', 191787.16],
  ['PALTAS', 184340.82],
  ['SARAGURO', 180366.38]],
 'memoria_driver': '2g',
 'rss_pico_mb': 2691}

In [6]:

# Extrapolación (regla de tres con el ratio medido) para el escenario nacional real
b_por_veh = ob.disco_mb.sum() * 1e6 / 2_821_910                    # bytes en disco por vehículo (6 tablas)
esc = []
for anios in (1, 5, 10):
    disco = 3_310_000 * anios * b_por_veh / 1e9                     # parque 2025 = 3,31 millones (INEC)
    esc.append({"escenario": f"parque nacional, {anios} año(s) de historia", "disco_gb": round(disco, 1),
                "ram_pandas_gb": round(disco * RES["mw"]["ratio_obj"], 1), "ram_pico_join_gb": round(disco * RES["mw"]["pico_max_ratio"], 1),
                "particiones_spark_128MB": int(np.ceil(disco * 1e3 / 128))})
esc = guardar(pd.DataFrame(esc), "05_extrapolacion"); RES["esc"] = esc.to_dict("records")
json.dump(RES, open(f"{TAB}/resumen_memory_wall.json", "w"), ensure_ascii=False, indent=1, default=str)
esc


,escenario,disco_gb,ram_pandas_gb,ram_pico_join_gb,particiones_spark_128MB
0,"parque nacional, 1 año(s) de historia",2.5,8.6,11.6,20
1,"parque nacional, 5 año(s) de historia",12.6,43.2,58.2,99
2,"parque nacional, 10 año(s) de historia",25.3,86.4,116.5,198
